In [ ]:
# import libraries
import requests # for making API requests
import pandas as pd # for data manipulation
from datetime import datetime # for handling dates and times
from zoneinfo import ZoneInfo # for timezone handling
import html # for HTML entity decoding

In [ ]:
# default parameters
limit = 10000 # max number of results per request
API_URL = 'https://api.openmeasures.io/content'

In [ ]:
# columns to keep after pulling requests for bluesky results
bluesky_columns_to_keep = [
    '_index', '_id', '_source.author', '_source.createdAt', '_source.text', '_source.embed.external.description', '_source.embed.external.title', '_source.authorProfile.handle', '_source.authorProfile.display_name', '_source.authorProfile._id', '_source.authorProfile.did'
]

# columns to keep after pulling requests for truth results
truth_columns_to_keep = [
    '_source.account.display_name', '_source.account.username', '_source.account.id', '_source.content_cleaned', '_source.created_at', '_source.datatype', '_source.mentions', '_id', '_source.in_reply_to_account_id', '_source.in_reply_to_id', '_source.reblog.id', '_source.replies_count', '_source.reblogs_count'
]

In [ ]:
"""
add functionality for getting search terms from user input, including desired timezone (default to user's local timezone) 

notes:
    * for truth social, the `site` param is `truth_social`
    * for bluesky, the `site` param is `bluesky`
    * the latest value the `until` param can take is a value at least 6 months prior to the current date

returns:
* params: a list of the search terms (list), including site, term, since, until, limit, querytype
* site term (str): the site being searched for site-specific functions
* time_field (str): the field to use for updating the 'since' parameter (e.g., 'created_at' or 'createdAt')

notes:
    * if the site is `bluesky`, the time_field is `createdAt`
    * if the site is `truth_social`, the time_field is `created_at`
"""

In [ ]:
"""
function to clean text by decoding HTML entities and handling encoding issues

params: text (str): the text to be cleaned
returns: cleaned text (str)
"""
def clean_text(text):
    if text:
        try:
            text = html.unescape(text) # decode HTML entities
            return text.encode('latin1').decode('utf-8') # encode to latin1 and decode to utf-8
        except (UnicodeDecodeError, UnicodeEncodeError):
            text = text.replace('\u2019', "’")
    return text

In [ ]:
"""
a function that continues to make requests to the API for the desired data until there are no more results to return OR API request limit is reached

params:
* api_url (str): the base URL for the API
* params (list): the parameters for the API request
* field (str): the field to use for updating the 'since' parameter (e.g., 'created_at' or 'createdAt')

TO-DO: add a way for the user to approve continuing requests if the single search limit of 10k results is reached AND return to the user how "far" the results have gotten in number of hours completed based off last timestamp in results against the original `until` param

NOTES:
- if the limit of 10k is reached, the user will get a message the *first* time and the user will get the option to discard, export the first 10k rows, or to complete more queries until either the search is complete OR the API request limit is reached
"""
def get_data(api_url, params, time_field) -> list:
    all_hits = []
    params_copy = params.copy()  # avoid modifying the original params

    while True:
        try:
            # api request
            response = requests.get(api_url, params=params_copy)
            response.raise_for_status()  # error for any http issues
            data = response.json()

            # Extract hits
            hits = data.get('hits', {}).get('hits', [])
            if not hits:  # break if no results
                break

            all_hits.extend(hits)

            # break if the number of hits is less than the limit
            if len(hits) < 10000:
                break

            # update the 'since' parameter with the last timestamp value
            last_created_at = hits[-1]['_source'].get(time_field)
            if not last_created_at:
                print("No 'created_at' found in the last hit.")
                break
            print(last_created_at)
            params_copy['since'] = last_created_at

        except Exception as e:
            print(f"Error occurred: {e}")
            break  # exit the loop on error

    return all_hits

In [ ]:
"""
function for cleaning text in the returned json results based on the source (bluesky or truth social)

params:
* hits (list): json array of results from the API
* site (str): the site being searched
"""
def clean_results(hits, site):
    if site == 'bluesky':
        for hit in hits:
            source = hit['_source']
            if source.get('embed') and source['embed'].get('external'):
                if source['embed']['external'].get('description') is not None:
                    source['embed']['external']['description'] = clean_text(source['embed']['external']['description'])
                if source['embed']['external'].get('title') is not None:
                    source['embed']['external']['title'] = clean_text(source['embed']['external']['title'])
            if source.get('text') is not None:
                source['text'] = clean_text(source['text'])
    elif site == 'truth_social':
        for hit in hits:
            source = hit['_source']
            if source.get('content_cleaned') is not None:
                source['content_cleaned'] = clean_text(source['content_cleaned'])
            if source.get('account') and source['account'].get('display_name') is not None:
                source['account']['display_name'] = clean_text(source['account']['display_name'])
            if source.get('account') and source['account'].get('username') is not None:
                source['account']['username'] = clean_text(source['account']['username'])

In [ ]:
"""
function for cleaning json results, converting to dataframe, and keeping only relevant columns

params:
* hits (list): json array of results from the API
* site (str): the site being searched

returns:
* df (dataframe): cleaned dataframe with only relevant columns for the user

TO-DO: convert timestamps to the timezone that user specified
"""
def process_results(hits, site):
    # clean text in results
    clean_results(hits, site)

    # convert to dataframe
    df = pd.json_normalize(hits)

    # keep only relevant columns
    if site == 'bluesky':
        df = df[bluesky_columns_to_keep]
    elif site == 'truth_social':
        df = df[truth_columns_to_keep]

    # drop '_source' prefix from column names
    df.columns = df.columns.str.replace('_source.', '', regex=False)

    return df

In [ ]:
"""
add functionality for saving results to a file (csv, excel, json)
"""

** add in function that picks up last datetime stamp from "past incomplete extractions"

** need to be able to save past search information

** future: "don't ask again" checkbox
